# LearnLens: data, computation and inductive bias
Run beside the learnlens/ package, or upload the release ZIP as learnlens.zip to Colab. CPU only. The default is a small controlled experiment; expand grids and seeds for research conclusions.


In [ ]:
import importlib.util, subprocess, sys
packages = {"numpy":"numpy", "scipy":"scipy", "sklearn":"scikit-learn", "matplotlib":"matplotlib", "pandas":"pandas"}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [ ]:
from pathlib import Path
import zipfile, sys
if not Path("learnlens/__init__.py").exists():
    archive = Path("learnlens.zip")
    if not archive.exists():
        raise FileNotFoundError("Place notebook beside learnlens/ or upload the release ZIP as learnlens.zip")
    root = Path.cwd().resolve()
    with zipfile.ZipFile(archive) as z:
        if not all((root / p).resolve().is_relative_to(root) for p in z.namelist()):
            raise ValueError("Unsafe ZIP path")
        z.extractall(root)
sys.path.insert(0, str(Path.cwd()))
from learnlens.benchmark import run_benchmark
print("LearnLens imported successfully")


## Fixed optimization budget, independent validation and test samples
SGD and the exact solver share training data. Crossings use validation only. Unobserved transitions remain censored. This is an exploratory experiment, not a proof of hardness.


In [ ]:
result = run_benchmark(dimensions=(12,), ks=(2,3), ns=(64,256,1024), seeds=(0,1), updates=400, output="outputs/notebook_benchmark.json")
assert len(result["records"]) == 24
print("Completed", len(result["records"]), "paired fits")


In [ ]:
import pandas as pd
frame = pd.DataFrame(result["records"])
print(frame.groupby(["k", "n", "learner"]).err_test.mean().unstack().to_string())
print(pd.DataFrame(result["transitions"]).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for k,ax in zip((2,3),axes):
    part=frame[frame.k==k]
    for learner in ("SGD","GF2"):
        curve=part[part.learner==learner].groupby("n").err_test.mean()
        ax.plot(curve.index,curve.values,"o-",label=learner)
    ax.set(xscale="log",xlabel="Training samples",title=f"Parity k={k}",ylim=(-.02,.6))
    ax.axhline(.5,linestyle="--",color="gray")
    ax.legend()
axes[0].set_ylabel("Mean independent test error")
fig.tight_layout()
plt.savefig("outputs/notebook_benchmark.png",dpi=150)
plt.show()
